In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,roc_curve

In [19]:
# Read the df and Split the data into train and test sets
df = pd.read_csv("../Data/cs-training.csv")
pd.set_option('float_format','{:3f}'.format)
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [20]:
#Median
income_median = X_train['MonthlyIncome'].median()

income_median = X_train['MonthlyIncome'].median()
temp_income = X_train['MonthlyIncome'].fillna(income_median)

#Cap

income_cap = temp_income.quantile(0.99)

In [21]:
# feature engineering
def transform(data_frame, income_median, income_cap):

    df = data_frame.copy()


    df['IncomeInformed'] = np.where(df['MonthlyIncome'].isnull(), 0, 1)


    df['MonthlyIncome_imp'] = df['MonthlyIncome'].fillna(income_median)


    df['MonthlyIncome_w'] = df['MonthlyIncome_imp'].clip(upper=income_cap)


    df['MonthlyIncome_log'] = 0.0

    mask = df['MonthlyIncome_w'] > 0

    df.loc[mask, 'MonthlyIncome_log'] = np.log(df.loc[mask, 'MonthlyIncome_w'])

    df['IncomexInformed'] = (df['MonthlyIncome_log']*df['IncomeInformed'])

    df['DebtRatio_Caped'] = np.where(df['DebtRatio']>1,1,df['DebtRatio'])

    df['NumberOfDependents_filtered'] = df['NumberOfDependents'].fillna(0)



    return df

In [22]:
#Transformation Train Test
X_train_prep = transform(X_train, income_median, income_cap)
X_test_prep  = transform(X_test, income_median, income_cap)

In [23]:
# Features
features =[
    'IncomexInformed',
    'DebtRatio_Caped',
    'NumberOfTimes90DaysLate',
    'age',
    'NumberOfOpenCreditLinesAndLoans']


In [24]:
#Model
X_train_model = X_train_prep[features]
X_test_model  = X_test_prep[features]

In [25]:
# logistic regression
model = LogisticRegression(
    max_iter=1000,
    solver = 'lbfgs'
)
model.fit(X_train_model, y_train)
y_pred_proba = model.predict_proba(X_test_model)[:,1]

In [26]:
#AUC
auc = roc_auc_score(y_test, y_pred_proba)
print(auc)

0.6620823417420276


In [27]:
#Ranking by Coeficient
coef_df = pd.DataFrame({
    'Variable': features,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient')

coef_df


,Variable,Coefficient
3,age,-0.031964
4,NumberOfOpenCreditLinesAndLoans,-0.024115
2,NumberOfTimes90DaysLate,0.039881
0,IncomexInformed,0.101152
1,DebtRatio_Caped,1.160296


In [28]:
#Creating the KS
ks_df = pd.DataFrame({
    'score': y_pred_proba,
    'target': y_test
})
ks_df = ks_df.sort_values(by='score', ascending=False)

#Creating the bad and good
ks_df['bad'] = ks_df['target']
ks_df['good'] = 1 - ks_df['target']

#Made a Cumulative
ks_df['cum_good'] = ks_df['good'].cumsum() / ks_df['good'].sum()
ks_df['cum_bad'] = ks_df['bad'].cumsum() / ks_df['bad'].sum()
ks_df['ks'] = abs(ks_df['cum_bad'] - ks_df['cum_good'])
ks_value = ks_df['ks'].max()
ks_value

np.float64(0.238604978773291)

#Making the ROC Curve grafic
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
auc_value = roc_auc_score(y_test, y_pred_proba)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc_value:.2f})')
plt.plot([0,1], [0,1], linestyle='--', color='gray', label='Random Model')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve – Credit Risk Model')
plt.legend()
plt.grid(True)
plt.savefig('../Images/ROC_Curve.png')
plt.show()